In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

#define names
daily_metrics_table = "coinwatch.gold.daily_metrics"
silver_table = "coinwatch.silver.candles"

volatility_table = "coinwatch.gold.asset_volatility"
volume_sessions_table = "coinwatch.gold.volume_sessions"
completeness_table = "coinwatch.gold.data_completeness"

In [0]:
%sql
select * from coinwatch.gold.daily_metrics limit(10)

In [0]:
asset_volatility_df = spark.sql("""
With vol as  (
    Select
        symbol,
        date,
        day_return_pct,
    -- last 7 day value
        stddev_samp(day_return_pct) over (
            Partition by symbol
            order by date
            rows between 6 preceding and current row
        ) as volatility_7d,

        -- count of rows in same window
        count(*) over (
            Partition by symbol
            order by date
        rows between 6 preceding and current row
        ) as days_in_window
    From coinwatch.gold.daily_metrics
),

bounds as (
    select
        symbol,
        percentile_cont(0.25)
            within group ( order by volatility_7d) as p25,
        percentile_cont(0.75) 
            WITHIN GROUP (ORDER BY volatility_7d) AS p75
    from vol 
    where days_in_window = 7
    group by symbol
)

select
    v.symbol,
    v.date,
    v.day_return_pct,
    v.volatility_7d,
    case
        when v.days_in_window < 7
            Then 'insufficient_history'
        when v.volatility_7d >= b.p75
            then 'high'
        when v.volatility_7d <= b.p25
            then 'low'
        ELSE 'normal' 
        END AS volatility_regime
from vol v
left join bounds b on v.symbol = b.symbol
""")

In [0]:
#write to asset.volitty table
(
    asset_volatility_df.write
    .format("delta")
    .mode("overwrite") #change to merge when to manydata, but other metrics also need to be changed
    .option("overwriteSchema","true")
    .saveAsTable("coinwatch.gold.asset_volatility")
)

In [0]:
%sql
select * from coinwatch.gold.asset_volatility limit(10)